# 模型推理与土地覆盖地图生成

**目标**: 使用已经训练好的ViT模型，对Part 1中生成的所有64x64图像块进行分类，并将分类结果拼接成一张可视化的土地覆盖地图。

**流程**: 
1.  加载与训练时完全相同的模型结构，并载入已保存的模型权重。
2.  定义一个自定义的数据集（Dataset）来加载Part 1中生成的所有图像块。
3.  对所有图像块执行推理（Inference），得到每个块的分类标签。
4.  根据图像块的文件名中包含的坐标信息，将分类结果重新组合（Reconstruct）成一个二维地图。
5.  为不同类别分配颜色，将最终的分类地图可视化并保存。
6.  (新增) 加载真实标签地图，生成并可视化混淆矩阵以评估模型性能。

### 步骤 1: 导入必要的库并设置环境

In [1]:
import torch
import torch.nn as nn
from torchvision.models import vit_b_32 # 确保使用与训练时相同的模型结构
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

# 新增导入：用于生成混淆矩阵
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

In [2]:
# 新增代码：解决 Matplotlib 显示中文乱码的问题
plt.rcParams['font.sans-serif'] = ['SimHei']  # 指定默认字体为黑体
plt.rcParams['axes.unicode_minus'] = False  # 解决保存图像是负号'-'显示为方块的问题

### 步骤 2: 加载模型

我们将实例化一个与你训练时**完全相同**的`vit_b_32`模型结构，然后加载你保存的`.pth`权重文件。

In [3]:
# --- 用户配置区域 ---

# 1. 模型权重文件所在的路径
models_path = "F:/vscode project/IMA_PW3/casestudy_pred/"

# 2. 你训练好的模型文件名 (不含.pth后缀)
model_name = "ViT32-Pretrained"

# --- 用户配置区域 ---

# # 1. 模型权重文件所在的路径
# models_path = "./models/"

# # 2. 你训练好的模型文件名 (不含.pth后缀)
# # ！！重要：根据错误日志，你的模型名应该是基于 vit_l_32 的
# model_name = "ViT32"  # <--- 请确认这是你训练 vit_l_32 时使用的模型名

# --- 配置结束 ---

from collections import OrderedDict
from torchvision.models import vit_l_32 # <--- 关键改动：改回 vit_l_32

# 确定设备
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"将使用设备: {DEVICE}")

# 标签映射
index_to_label = {
    0: 'AnnualCrop', 1: 'Forest', 2: 'HerbaceousVegetation', 3: 'Highway', 
    4: 'Industrial', 5: 'Pasture', 6: 'PermanentCrop', 7: 'Residential', 
    8: 'River', 9: 'SeaLake'
}
num_classes = len(index_to_label)

# 实例化与训练时相同的模型结构 (vit_l_32)
model = vit_l_32()
# 大型版(Large)的 head 输入维度是 1024
model.heads.head = nn.Linear(1024, num_classes) # <--- 关键改动：维度从 768 改为 1024

# 构建模型权重文件的完整路径
model_save_path = os.path.join(models_path, model_name + ".pth")

try:
    state_dict = torch.load(model_save_path, map_location=DEVICE)
    # 如果你的权重文件仍然带有 'module.' 前缀，保留这部分清理代码
    if list(state_dict.keys())[0].startswith('module.'):
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k[7:]
            new_state_dict[name] = v
        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(state_dict)
    
    print(f"成功加载模型权重: {model_save_path}")

except FileNotFoundError:
    print(f"错误: 找不到模型文件 '{model_save_path}'。请检查路径和文件名。")
except Exception as e:
    print(f"加载模型时发生错误: {e}")

model.to(DEVICE)
model.eval()

将使用设备: cuda:0
成功加载模型权重: F:/vscode project/IMA_PW3/casestudy_pred/ViT32-Pretrained.pth


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 2.00 GiB of which 0 bytes is free. Of the allocated memory 1.71 GiB is allocated by PyTorch, and 1.91 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### 步骤 3: 准备数据加载器

我们需要定义一个自定义的 `Dataset` 类，它能够读取 `./patches_64x64/` 文件夹中的所有图像块，并应用与训练时完全相同的预处理变换。同时，我们还需要解析文件名来获取每个块的坐标。

In [4]:
# 1. 定义与训练时完全相同的图像预处理流程
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. 创建自定义数据集类 (这部分代码无需修改)
class PatchDataset(Dataset):
    # ... (代码与之前相同) ...
    def __init__(self, patch_dir, transform=None):
        self.patch_dir = patch_dir
        self.transform = transform
        self.patch_files = sorted([f for f in os.listdir(patch_dir) if f.endswith('.png')], 
                                  key=lambda x: (int(x.split('_')[1]), int(x.split('_')[2].split('.')[0])))

    def __len__(self):
        return len(self.patch_files)

    def __getitem__(self, idx):
        img_name = self.patch_files[idx]
        img_path = os.path.join(self.patch_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        parts = img_name.split('.')[0].split('_')
        row = int(parts[1])
        col = int(parts[2])
        
        return image, (row, col)

# 3. 实例化Dataset和DataLoader
patches_dir = "./patches_64x64/"
patch_dataset = PatchDataset(patch_dir=patches_dir, transform=inference_transform)

# ！！！关键改动：大幅减小批量大小以适应 vit_l_32 模型！！！
batch_size = 8  # <--- 从 64 减小到 8。如果仍然内存溢出，可以尝试 4 或 2。

patch_loader = DataLoader(patch_dataset, batch_size=batch_size, shuffle=False, num_workers=0) # 设置 num_workers=0 以简化调试

print(f"找到了 {len(patch_dataset)} 个图像块，将以 {batch_size} 的批量大小进行处理。")
print("每个图像块在输入模型前都将被调整为 224x224 尺寸。")

### 步骤 4: 执行推理并构建分类地图

现在，我们遍历所有数据，用模型进行预测，并将结果填充到一个代表最终地图的二维数组中。

In [5]:
# 1. 确定地图的最终尺寸
max_row = 0
max_col = 0
for filename in patch_dataset.patch_files:
    parts = filename.split('.')[0].split('_')
    max_row = max(max_row, int(parts[1]))
    max_col = max(max_col, int(parts[2]))

map_height = max_row + 1
map_width = max_col + 1
classification_map = np.zeros((map_height, map_width), dtype=np.int32)

print(f"将要生成的分类地图尺寸为: {map_height} x {map_width}")

将要生成的分类地图尺寸为: 34 x 77


### 步骤 4 (补充): 加载真实的土地覆盖标签 (Ground Truth)

为了生成混淆矩阵，我们需要将模型的预测结果与真实的标签进行对比。这里，我们假设你有一个与最终输出地图尺寸完全相同的“真实标签地图”。这个地图的每个像素值代表了该地块的真实类别索引。

**请将 `ground_truth_map_path` 变量修改为你真实的标签文件名。**

In [6]:
# --- 用户配置 ---
# 请将 'path/to/your/ground_truth_map.png' 替换为你的真实标签地图文件路径
# 例如: ground_truth_map_path = "./ground_truth_map.png"
# 重要提示：该地图的尺寸必须是 34x77 (与上面计算出的 map_height x map_width 相同)
ground_truth_map_path = "path/to/your/ground_truth_map.png"
# --- 配置结束 ---

try:
    # 加载真实标签地图 (通常是单通道的灰度图)
    ground_truth_image = Image.open(ground_truth_map_path)
    # 将其转换为NumPy数组
    ground_truth_map = np.array(ground_truth_image)

    # 检查尺寸是否匹配
    if ground_truth_map.shape != (map_height, map_width):
        print(f"警告: 真实标签地图的尺寸 {ground_truth_map.shape} 与预测地图的尺寸 {(map_height, map_width)} 不匹配。")
        print("混淆矩阵的结果可能不准确。请确保两个地图的尺寸完全相同。")
    else:
        print(f"成功加载真实标签地图: {ground_truth_map_path}, 尺寸: {ground_truth_map.shape}")

except FileNotFoundError:
    print(f"错误: 找不到真实标签地图文件 '{ground_truth_map_path}'。")
    print("将无法生成混淆矩阵。请检查文件路径。")
    ground_truth_map = None
except Exception as e:
    print(f"加载真实标签地图时发生错误: {e}")
    ground_truth_map = None

In [7]:
# 2. 循环执行推理
with torch.no_grad(): # 禁用梯度计算以加速并节省内存
    for images, coords in tqdm(patch_loader, desc="正在分类"):
        images = images.to(DEVICE)
        
        # 模型预测
        outputs = model(images)
        _, predicted_indices = torch.max(outputs, 1)
        
        # 将预测结果移回CPU
        predictions_np = predicted_indices.cpu().numpy()
        
        # 将结果填充到地图上
        rows, cols = coords
        for i in range(len(predictions_np)):
            classification_map[rows[i], cols[i]] = predictions_np[i]

print("\n分类地图已在内存中生成。")

正在分类:  39%|███▉      | 16/41 [00:50<01:18,  3.13s/it]


KeyboardInterrupt: 

### 步骤 5: 可视化土地覆盖地图

最后一步是为地图上的每个类别索引分配一种颜色，并使用 `matplotlib` 将其显示出来。我们还会添加一个图例，以便解读地图。

In [8]:
import matplotlib.patches as mpatches

# 1. 定义一个颜色映射 (RGB格式, 0-255)
color_map = {
    0: [255, 255, 0],    # AnnualCrop (黄色)
    1: [0, 128, 0],      # Forest (深绿)
    2: [152, 251, 152],  # HerbaceousVegetation (浅绿)
    3: [128, 128, 128],  # Highway (灰色)
    4: [139, 0, 0],      # Industrial (深红)
    5: [255, 165, 0],    # Pasture (橙色)
    6: [210, 105, 30],   # PermanentCrop (巧克力色)
    7: [255, 0, 0],      # Residential (红色)
    8: [0, 0, 255],      # River (蓝色)
    9: [0, 191, 255]     # SeaLake (深天蓝)
}

# 2. 将分类地图（整数）转换为RGB图像（三维数组）
rgb_map = np.zeros((map_height, map_width, 3), dtype=np.uint8)
for i in range(map_height):
    for j in range(map_width):
        class_index = classification_map[i, j]
        rgb_map[i, j] = color_map[class_index]

# 3. 使用matplotlib进行可视化
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(rgb_map)
ax.set_title("土地覆盖分类地图 (Land Cover Classification Map)", fontsize=16)
ax.axis('off') # 不显示坐标轴

# 4. 创建并显示图例
legend_patches = [mpatches.Patch(color=np.array(c)/255., label=l) 
                  for l, c in zip(index_to_label.values(), color_map.values())]

ax.legend(handles=legend_patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

plt.tight_layout()
plt.show()

# 5. (可选) 将最终地图保存为图像文件
result_image = Image.fromarray(rgb_map)
map_filename = "land_cover_map_64x64.png"
result_image.save(map_filename)
print(f"\n最终的分类地图已保存为: '{map_filename}'")


最终的分类地图已保存为: 'land_cover_map_64x64.png'


### 步骤 6: (新增) 生成并可视化混淆矩阵

现在我们有了模型的预测结果 (`classification_map`) 和真实的标签 (`ground_truth_map`)，我们可以计算并展示混淆矩阵，以评估模型在每个类别上的表现。

In [9]:
if ground_truth_map is not None:
    # 1. 将二维的地图展平为一维数组，以便进行比较
    y_true = ground_truth_map.flatten()
    y_pred = classification_map.flatten()
    
    # 2. 获取所有类别的名称
    class_labels = list(index_to_label.values())
    
    # 3. 计算混淆矩阵
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_labels)))
    
    print("\n--- 混淆矩阵 ---")
    print(cm)
    
    # 4. 使用 Seaborn Heatmap进行可视化
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title('混淆矩阵 (Confusion Matrix)', fontsize=16)
    plt.ylabel('真实类别 (True Label)', fontsize=12)
    plt.xlabel('预测类别 (Predicted Label)', fontsize=12)
    plt.xticks(rotation=45, ha='right') # 旋转x轴标签以防重叠
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
else:
    print("\n由于未能加载真实标签地图，跳过生成混淆矩阵的步骤。")